<a href="https://colab.research.google.com/github/maureljean55/atelier-saint-honore/blob/main/notebooks/02_q1_q2_descriptif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
clients = pd.read_csv("client_propre.csv")
boutiques = pd.read_csv("boutiques_propre.csv")

print("clients :", clients.shape)
print("boutiques :", boutiques.shape)

clients : (679, 13)
boutiques : (24, 5)


In [31]:
clients["evolution_panier_pct"] = (clients["montant_prochain_achat_eur"] / clients["panier_moyen_eur"] - 1) * 100

clients[["client_id", "panier_moyen_eur", "montant_prochain_achat_eur", "evolution_panier_pct"]].head()

,client_id,panier_moyen_eur,montant_prochain_achat_eur,evolution_panier_pct
0,CT80059,1468.0,725.0,-50.613079
1,CT80578,1235.0,1479.0,19.757085
2,CT80256,1814.0,2307.0,27.177508
3,CT80098,1795.0,1820.0,1.392758
4,CT80362,2022.0,1201.0,-40.603363


In [32]:
print("Médiane de l'évolution :", round(clients["evolution_panier_pct"].median(), 1), "%")
print("Moyenne de l'évolution :", round(clients["evolution_panier_pct"].mean(), 1), "%")

part_en_baisse = (clients["evolution_panier_pct"] < 0).mean() * 100
print("Part des clients dont le panier baisse :", round(part_en_baisse, 1), "%")

Médiane de l'évolution : -23.2 %
Moyenne de l'évolution : -14.9 %
Part des clients dont le panier baisse : 76.7 %


In [33]:
boutiques["ville"] = boutiques["ville"].str.strip().str.lower()

pays_des_villes = {
    "paris": "France",
    "cannes": "France",
    "bordeaux": "France",
    "dubaï": "EAU",
    "shanghai": "Chine",
    "séoul": "Corée",
    "genève": "Suisse",
    "new york": "USA",
    "londres": "Royaume-Uni"
}

boutiques["pays_boutique"] = boutiques["ville"].map(pays_des_villes)

print("Villes sans pays :", boutiques["pays_boutique"].isna().sum())
boutiques[["ville", "pays_boutique"]].drop_duplicates()

Villes sans pays : 0


,ville,pays_boutique
0,dubaï,EAU
1,shanghai,Chine
2,cannes,France
4,genève,Suisse
5,séoul,Corée
6,paris,France
10,bordeaux,France
11,new york,USA
15,londres,Royaume-Uni


In [34]:
clients = clients.merge(
    boutiques[["boutique_id", "ville", "pays_boutique"]],
    left_on="boutique_principale",
    right_on="boutique_id",
    how="left"
)
clients = clients.drop(columns="boutique_id")

print(clients.shape)
clients[["client_id", "boutique_principale", "ville", "pays_boutique", "pays_residence"]].head()

(679, 16)


,client_id,boutique_principale,ville,pays_boutique,pays_residence
0,CT80059,BT15,séoul,Corée,usa
1,CT80578,BT28,cannes,France,france
2,CT80256,BT27,dubaï,EAU,france
3,CT80098,BT16,paris,France,italie
4,CT80362,BT31,dubaï,EAU,chine


In [35]:
clients["touriste"] = clients["pays_residence"].str.lower() != clients["pays_boutique"].str.lower()

clients["touriste"].value_counts()

,count
touriste,
True,533
False,146


In [36]:
clients.groupby("touriste")["evolution_panier_pct"].agg(["median", "mean", "count"]).round(1)

,median,mean,count
touriste,,,
False,-21.9,-13.8,146
True,-23.6,-15.2,533


In [37]:
clients.groupby("canal_prefere")["evolution_panier_pct"].agg(["median", "mean", "count"]).round(1)

,median,mean,count
canal_prefere,,,
boutique,-23.1,-16.7,390
en ligne,-23.8,-12.8,154
mixte,-22.3,-12.1,135


In [38]:
clients.groupby("carte_fidelite")["evolution_panier_pct"].agg(["median", "mean", "count"]).round(1)

,median,mean,count
carte_fidelite,,,
argent,-27.8,-22.7,242
aucune,-26.9,-20.7,336
or,9.6,23.1,101


In [39]:
clients.head()

,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m,montant_total_eur,evolution_panier_pct,ville,pays_boutique,touriste
0,CT80059,BT15,4,4,1468.0,0,0,aucune,en ligne,usa,725.0,1,5872.0,-50.613079,séoul,Corée,True
1,CT80578,BT28,80,3,1235.0,1,1,argent,boutique,france,1479.0,1,3705.0,19.757085,cannes,France,False
2,CT80256,BT27,43,1,1814.0,0,2,or,boutique,france,2307.0,1,1814.0,27.177508,dubaï,EAU,True
3,CT80098,BT16,1,1,1795.0,0,0,or,boutique,italie,1820.0,0,1795.0,1.392758,paris,France,True
4,CT80362,BT31,51,3,2022.0,0,0,argent,boutique,chine,1201.0,1,6066.0,-40.603363,dubaï,EAU,True


In [40]:
seuil = clients["montant_total_eur"].quantile(0.75)
clients["gros_depensier"] = clients["montant_total_eur"] >= seuil

resultat = clients.groupby("gros_depensier").agg(
    nb_retour=("est_revenu_6m", "count"),
    taux_retour = ("est_revenu_6m", "mean"))

resultat["taux_retour"] = resultat["taux_retour"] * 100
print(resultat)



                nb_retour  taux_retour
gros_depensier                        
False                 509    35.952849
True                  170    55.882353


In [41]:
clients.groupby("gros_depensier").agg(
    nb_achat =("nb_achats_12m", "count"),
    moyenne_achat = ("nb_achats_12m", "mean"),
    median_achat = ("nb_achats_12m", "median"))


,nb_achat,moyenne_achat,median_achat
gros_depensier,,,
False,509,2.430255,2.0
True,170,7.323529,7.0


In [42]:
clients.groupby("gros_depensier").agg(
    anciennete_moyen =("anciennete_mois", "mean"),
    mois_median = ("anciennete_mois", "median"))




,anciennete_moyen,mois_median
gros_depensier,,
False,32.469548,26.0
True,26.047059,18.5


In [43]:
clients.groupby("gros_depensier").agg(
    prochainMontant_moyen =("montant_prochain_achat_eur", "mean"),
    prochainMontant_median = ("montant_prochain_achat_eur", "median"))



,prochainMontant_moyen,prochainMontant_median
gros_depensier,,
False,1583.605108,1502.0
True,3150.105882,2831.0


In [44]:
cols = ["montant_total_eur","nb_achats_12m","est_revenu_6m","anciennete_mois","montant_prochain_achat_eur"]
clients[cols].corr()

,montant_total_eur,nb_achats_12m,est_revenu_6m,anciennete_mois,montant_prochain_achat_eur
montant_total_eur,1.000000,0.837919,0.173428,-0.213705,0.784193
nb_achats_12m,0.837919,1.000000,0.209154,-0.046282,0.530307
est_revenu_6m,0.173428,0.209154,1.000000,0.116429,0.135827
anciennete_mois,-0.213705,-0.046282,0.116429,1.000000,-0.287189
montant_prochain_achat_eur,0.784193,0.530307,0.135827,-0.287189,1.000000
